# Phi1-Phi2 Data Analysis: Python Recreation of GNUplot Script

This notebook recreates the functionality of the `plot_SS_2.cmd` GNUplot script. It generates a multiplot layout with:
- **Top row**: Three 2D heatmaps showing different projections of the phi1-phi2 data
- **Bottom row**: Three 1D line plots showing cross-sections where phi2=0

The data file `phi1_phi2_eBest_SS.dat` contains 5 columns:
1. phi1 (angle 1)
2. phi2 (angle 2) 
3. Column 3 (z-data)
4. Column 4 (data)
5. Column 5 (energy data)

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib parameters for better display
plt.rcParams['figure.figsize'] = (18, 10)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True

print("Libraries imported successfully!")

In [ ]:
# Load and Prepare Data
filename = "phi1_phi2_eBest_SS.dat"

# Load data using pandas
# Columns: phi1, phi2, col3, col4, col5
data = pd.read_csv(filename, sep='\s+', header=None, 
                   names=['phi1', 'phi2', 'col3', 'col4', 'col5'])

print(f"Data loaded: {len(data)} rows, {len(data.columns)} columns")
print(f"Phi1 range: {data['phi1'].min()} to {data['phi1'].max()}")
print(f"Phi2 range: {data['phi2'].min()} to {data['phi2'].max()}")
print("\nFirst few rows:")
print(data.head())

# Create pivot tables for heatmap plotting
phi1_unique = np.sort(data['phi1'].unique())
phi2_unique = np.sort(data['phi2'].unique())

print(f"\nUnique phi1 values: {len(phi1_unique)}")
print(f"Unique phi2 values: {len(phi2_unique)}")

# Create 2D arrays for plotting
def create_grid(data, col_name):
    """Create 2D grid from data for plotting"""
    return data.pivot(index='phi2', columns='phi1', values=col_name).values

# Create grids for each column
grid_col3 = create_grid(data, 'col3')
grid_col4 = create_grid(data, 'col4') 
grid_col5 = create_grid(data, 'col5')

print(f"Grid shape: {grid_col5.shape}")

In [ ]:
# Define Helper Functions (from GNUplot script)

# Constants from the original script
zmax = 1.39687500
energy_offset = -6227.1749

def shift_z(x):
    """Equivalent to GNUplot: shift_z(x)= (x < zmax/2) ? x : x-zmax"""
    return np.where(x < zmax/2, x, x - zmax)

def shift_phi(x):
    """Equivalent to GNUplot: shift_phi(x)= (x < 180) ? x : x-360"""
    return np.where(x < 180, x, x - 360)

# Test the functions
print("Testing helper functions:")
test_values = np.array([-180, -90, 0, 90, 180, 270])
print(f"Original values: {test_values}")
print(f"shift_phi result: {shift_phi(test_values)}")

test_z = np.array([0, 0.5, 0.7, 1.0, 1.4])
print(f"Original z values: {test_z}")
print(f"shift_z result: {shift_z(test_z)}")

In [ ]:
# Configure Plot Layout and Styling

# Create custom colormap for the third plot (from GNUplot palette)
# GNUplot palette: (0 '#FF0000' , 0.5 '#008800', 0.9 '#0000FF',1.0 '#FFFFFF',1.1 '#888800',1.5 '#FF00FF', 2 '#FF0000')
colors_custom = ['#FF0000', '#008800', '#0000FF', '#FFFFFF', '#888800', '#FF00FF', '#FF0000']
positions = [0.0, 0.25, 0.45, 0.5, 0.55, 0.75, 1.0]
cmap_custom = LinearSegmentedColormap.from_list('custom', list(zip(positions, colors_custom)))

# Set up coordinate arrays for plotting
X, Y = np.meshgrid(phi1_unique, phi2_unique)

# For shifted coordinates
X_shifted = shift_phi(X)
Y_shifted = shift_phi(Y)

# Grid lines setup (equivalent to GNUplot grid lines)
def add_grid_lines(ax, color='#CCCCCC', linestyle='--', alpha=0.7):
    """Add grid lines similar to GNUplot"""
    # Vertical lines
    for phi in [-130, 50, 230, -310]:
        if phi >= -180 and phi <= 180:
            ax.axvline(x=phi, color=color, linestyle=linestyle, alpha=alpha, linewidth=1)
    
    # Horizontal lines  
    for i in range(-170, 181, 20):
        if i >= -180 and i <= 180:
            ax.axhline(y=i, color=color, linestyle=linestyle, alpha=alpha, linewidth=1)

print("Plot configuration completed!")
print(f"Custom colormap created with {len(colors_custom)} colors")
print(f"Coordinate grids: X shape {X.shape}, Y shape {Y.shape}")

In [ ]:
# Create Complete Multiplot Layout (Top and Bottom Rows)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# =============================================================================
# TOP ROW: 2D Heatmaps - Equivalent to GNUplot 3D surface plots viewed from top
# =============================================================================

# Plot 1: Energy data (column 5 with offset)
energy_data = grid_col5 - 2 * energy_offset
im1 = axes[0,0].contourf(X, Y, energy_data, levels=50, cmap='viridis')
axes[0,0].set_title('Energy Data (Col5 - 2*offset)')
axes[0,0].set_xlabel('Phi1')
axes[0,0].set_ylabel('Phi2')
axes[0,0].set_xlim(-180, 180)
axes[0,0].set_ylim(-180, 180)
axes[0,0].grid(True, alpha=0.3)

# Add arrows (equivalent to GNUplot arrows)
axes[0,0].annotate('', xy=(120, -10), xytext=(-60, -10), 
                   arrowprops=dict(arrowstyle='-', color='#888888', lw=3))
axes[0,0].annotate('', xy=(140, 10), xytext=(-40, 10), 
                   arrowprops=dict(arrowstyle='-', color='#888888', lw=3))
axes[0,0].annotate('', xy=(-40, 10), xytext=(-60, -10), 
                   arrowprops=dict(arrowstyle='-', color='#888888', lw=3))
axes[0,0].annotate('', xy=(140, 10), xytext=(120, -10), 
                   arrowprops=dict(arrowstyle='-', color='#888888', lw=3))

plt.colorbar(im1, ax=axes[0,0])

# Plot 2: Column 4 data with shifted coordinates
im2 = axes[0,1].contourf(X_shifted, Y_shifted, grid_col4, levels=50, cmap='viridis')
axes[0,1].set_title('Column 4 Data (shifted coordinates)')
axes[0,1].set_xlabel('Phi1 (shifted)')
axes[0,1].set_ylabel('Phi2 (shifted)')
axes[0,1].set_xlim(-180, 180)
axes[0,1].set_ylim(-180, 180)
axes[0,1].grid(True, alpha=0.3)
plt.colorbar(im2, ax=axes[0,1])

# Plot 3: Column 3 data with custom colormap and shifted coordinates
grid_col3_shifted = shift_z(grid_col3)
im3 = axes[0,2].contourf(X_shifted, Y_shifted, grid_col3_shifted, levels=50, cmap=cmap_custom)
axes[0,2].set_title('Column 3 Data (shifted z and coordinates)')
axes[0,2].set_xlabel('Phi1 (shifted)')
axes[0,2].set_ylabel('Phi2 (shifted)')
axes[0,2].set_xlim(-180, 180)
axes[0,2].set_ylim(-180, 180)
axes[0,2].grid(True, alpha=0.3)
plt.colorbar(im3, ax=axes[0,2])

# =============================================================================
# BOTTOM ROW: 1D Line Plots - Cross-sections where phi2=0
# =============================================================================

# Extract data where phi2 = 0 (equivalent to GNUplot awk '{if ($2==0) print}')
data_phi2_zero = data[data['phi2'] == 0].copy()
data_phi2_zero = data_phi2_zero.sort_values('phi1')

print(f"Data points where phi2=0: {len(data_phi2_zero)}")

# Plot 1: Energy data cross-section
energy_1d = data_phi2_zero['col5'] - 2 * energy_offset
axes[1,0].plot(data_phi2_zero['phi1'], energy_1d, 'b-', linewidth=2)
axes[1,0].set_title('Energy Cross-section (phi2=0)')
axes[1,0].set_xlabel('Phi1')
axes[1,0].set_ylabel('Energy')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xlim(-180, 180)

# Plot 2: Column 4 cross-section
axes[1,1].plot(data_phi2_zero['phi1'], data_phi2_zero['col4'], 'b-', linewidth=2)
axes[1,1].set_title('Column 4 Cross-section (phi2=0)')
axes[1,1].set_xlabel('Phi1')
axes[1,1].set_ylabel('Column 4 Value')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_xlim(-180, 180)

# Plot 3: Column 3 (shifted) cross-section
col3_1d_shifted = shift_z(data_phi2_zero['col3'])
axes[1,2].plot(data_phi2_zero['phi1'], col3_1d_shifted, 'b-', linewidth=2)
axes[1,2].set_title('Column 3 (shifted) Cross-section (phi2=0)')
axes[1,2].set_xlabel('Phi1')
axes[1,2].set_ylabel('Shifted Column 3 Value')
axes[1,2].grid(True, alpha=0.3)
axes[1,2].set_xlim(-180, 180)

# =============================================================================
# DISPLAY COMPLETE PLOT
# =============================================================================

# Adjust layout and spacing
plt.tight_layout(pad=3.0)

# Add overall title
fig.suptitle('Phi1-Phi2 Data Analysis: Recreation of GNUplot Script', fontsize=16, y=0.95)

# Display the complete plot with both top and bottom rows
plt.show()

print("="*60)
print("COMPLETE MULTIPLOT CREATED SUCCESSFULLY!")
print("="*60)
print("✓ Top row: Three 2D heatmaps with different data projections")
print("✓ Bottom row: Three 1D line plots showing cross-sections")
print("✓ Custom color palette applied")
print("✓ Coordinate transformations (shift_phi, shift_z)")
print("✓ Grid lines and styling")
print("✓ Energy offset corrections applied")
print("="*60)

In [ ]:
# Optional: 3D Surface Visualization (Bonus - showing actual 3D surfaces)

fig = plt.figure(figsize=(18, 6))

# 3D Surface Plot 1: Energy data
ax1 = fig.add_subplot(131, projection='3d')
surf1 = ax1.plot_surface(X, Y, energy_data, cmap='viridis', alpha=0.8)
ax1.set_title('3D Surface: Energy Data')
ax1.set_xlabel('$\phi_1$')
ax1.set_ylabel('$\phi_2$')
ax1.set_zlabel('Energy')
ax1.view_init(elev=30, azim=45)

# 3D Surface Plot 2: Column 4 data
ax2 = fig.add_subplot(132, projection='3d')
surf2 = ax2.plot_surface(X_shifted, Y_shifted, grid_col4, cmap='viridis', alpha=0.8)
ax2.set_title('3D Surface: Column 4 Data (shifted)')
ax2.set_xlabel('$\phi_1$ (shifted)')
ax2.set_ylabel('$\phi_2$ (shifted)')
ax2.set_zlabel('Column 4')
ax2.view_init(elev=30, azim=45)

# 3D Surface Plot 3: Column 3 data with custom colormap
ax3 = fig.add_subplot(133, projection='3d')
surf3 = ax3.plot_surface(X_shifted, Y_shifted, grid_col3_shifted, cmap=cmap_custom, alpha=0.8)
ax3.set_title('3D Surface: Column 3 Data (shifted)')
ax3.set_xlabel('$\phi_1$ (shifted)')
ax3.set_ylabel('$\phi_2$ (shifted)')
ax3.set_zlabel('Column 3 (shifted)')
ax3.view_init(elev=30, azim=45)

plt.tight_layout()
plt.show()

print("3D surface plots created as bonus visualization!")